# Ablation Study: Pengaruh α terhadap ROUGE dan Entailment

Notebook ini menjalankan generate + NLI reranking dengan variasi α (0.0, 0.3, 0.5, 0.7, 1.0) untuk melihat dampaknya terhadap kualitas summary.

## 1. Cek GPU & Install

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Cek internet
!curl -s --max-time 5 https://huggingface.co > /dev/null && echo "Internet: ON" || echo "Internet: OFF"

!pip install -q transformers evaluate rouge-score

## 2. Cari File

Upload **model checkpoint** (dari hasil training sebelumnya) dan **test.jsonl** via Add Data.

In [ ]:
import os

print("=== Isi ./data/kaggle_input/ ===")
for root, dirs, files in os.walk("./data/kaggle_input/"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
MODEL_PATH = "./checkpoints/bart-baseline"
TEST_FILE = "./data/test.jsonl"

print("Model path:", MODEL_PATH)
print("Test file:", TEST_FILE)
print("config.json exists:", os.path.exists(os.path.join(MODEL_PATH, "config.json")))
print("test.jsonl exists:", os.path.exists(TEST_FILE))

## 3. Load Models

In [ ]:
import json
import math
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
TEXT_COLUMN = "article"
SUMMARY_COLUMN = "summary"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BART
bart_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device).eval()
print("BART loaded")

# Load NLI
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(device).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)
neu_idx = next(i for i, l in id2label.items() if "neutral" in l)
print("NLI loaded")

## 4. Generate Semua Kandidat (Sekali Saja)

Generate 4 kandidat per sample, simpan semua. Reranking dengan α berbeda dilakukan setelahnya tanpa perlu generate ulang.

In [ ]:
# Load test data
with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_rows = [json.loads(l) for l in f if l.strip()]

print(f"Test samples: {len(test_rows)}")

NUM_CANDIDATES = 4
all_candidates = []

for i, row in enumerate(test_rows):
    document = row[TEXT_COLUMN]

    # Generate candidates
    with torch.inference_mode():
        inputs = bart_tokenizer(document, truncation=True, max_length=256, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        generated = bart_model.generate(
            **inputs,
            max_length=128, min_length=32,
            num_beams=NUM_CANDIDATES, num_return_sequences=NUM_CANDIDATES,
            length_penalty=1.0, early_stopping=True,
            output_scores=True, return_dict_in_generate=True,
        )
    texts = bart_tokenizer.batch_decode(generated.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    scores = generated.sequences_scores.tolist()

    # NLI score each candidate
    candidates = []
    for text, gen_score in zip(texts, scores):
        summary = text.strip()
        with torch.inference_mode():
            enc = nli_tokenizer(document, summary, truncation=True, max_length=512, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            probs = torch.softmax(nli_model(**enc).logits[0], dim=-1)
        candidates.append({
            "summary": summary,
            "generation_score": float(gen_score),
            "entailment": float(probs[ent_idx]),
            "contradiction": float(probs[con_idx]),
            "neutral": float(probs[neu_idx]),
        })

    all_candidates.append({
        "id": row.get("id"),
        "document": document,
        "reference_summary": row.get(SUMMARY_COLUMN),
        "candidates": candidates,
    })

    if (i + 1) % 500 == 0:
        print(f"Generated {i+1}/{len(test_rows)}", flush=True)

print(f"\nDone. Generated candidates for {len(all_candidates)} samples.")

## 5. Reranking dengan Variasi α

In [ ]:
import statistics
import evaluate
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

ALPHA_VALUES = [0.0, 0.3, 0.5, 0.7, 1.0]
results = []

for alpha in ALPHA_VALUES:
    r1, r2, rl = [], [], []
    ent_scores, con_scores = [], []

    for item in all_candidates:
        # Rerank candidates with current alpha
        best = None
        for c in item["candidates"]:
            norm_gen = math.tanh(c["generation_score"] / 10.0)
            combined = alpha * c["entailment"] + (1 - alpha) * norm_gen
            if best is None or combined > best["combined"]:
                best = {**c, "combined": combined}

        # ROUGE
        s = scorer.score(item["reference_summary"], best["summary"])
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
        ent_scores.append(best["entailment"])
        con_scores.append(best["contradiction"])

    n = len(all_candidates)
    result = {
        "alpha": alpha,
        "rouge1": sum(r1) / n,
        "rouge2": sum(r2) / n,
        "rougeL": sum(rl) / n,
        "entailment": statistics.mean(ent_scores),
        "contradiction": statistics.mean(con_scores),
    }
    results.append(result)
    print(f"α={alpha:.1f} | R1={result['rouge1']:.4f} | R2={result['rouge2']:.4f} | RL={result['rougeL']:.4f} | Ent={result['entailment']:.4f} | Con={result['contradiction']:.4f}")

print("\nAblation selesai.")

## 6. Simpan Hasil

In [ ]:
output_path = Path("./results/ablation_alpha_results.json")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved to {output_path}")

# Tabel ringkasan
print("\n=== Ablation Study: α ===")
print(f"{'α':>5} | {'ROUGE-1':>8} | {'ROUGE-2':>8} | {'ROUGE-L':>8} | {'Entailment':>10} | {'Contradiction':>13}")
print("-" * 70)
for r in results:
    print(f"{r['alpha']:>5.1f} | {r['rouge1']:>8.4f} | {r['rouge2']:>8.4f} | {r['rougeL']:>8.4f} | {r['entailment']:>10.4f} | {r['contradiction']:>13.4f}")